# Step 5 verification and neural-model handover

Group 13 • prepared for Jiayi's assigned neural-model review • source revision `ced8091063968093c2841460a6f90ddf14e9979b`.

This notebook independently checks the **saved corrected experiment** and explains the implemented architecture. It does not train or modify a model, access raw user files, edit the report, or push files. The reported experiment used 20,000 sampled user records; this review reads all 266,640 held-out candidate predictions, including 3,000 query slates.

Completed checks: all four metrics for six variants; input-order invariance; an independent metric implementation; paired DHAR–MLP uncertainty; training-log checkpoint selection; horizon coverage; source/diagram consistency. A fresh training run and neural-checkpoint reload are **not** claimed. Large neural checkpoints are not committed to this repository.

Use the team's `Step5_Neural_Rankers_DHAR.ipynb` for training. This companion is intentionally lightweight and requires no Torch or GPU. Saved outputs below were executed locally against the pinned public repository. Human author review remains pending.

## 1 Setup and source provenance
The optional install cell is for a fresh Colab session. Use a new checkout of the pinned revision so later branch changes cannot silently alter this review. No credentials or private raw data are required.

In [1]:
# In a fresh Colab session, uncomment and run:
# %pip -q install numpy pandas "pyarrow>=17,<20"

In [2]:
from pathlib import Path
import sys, json, subprocess, ast, hashlib
import numpy as np
import pandas as pd
from IPython.display import display, SVG

PINNED_REVISION = 'ced8091063968093c2841460a6f90ddf14e9979b'
REPO_URL = 'https://github.com/steffanylars/nus-web-mining-business-insights.git'
# Override REVIEW_REPO before this cell to use an existing read-only checkout.
REVIEW_REPO = Path(globals().get('REVIEW_REPO', Path.cwd() / ('jdsearch_review_' + PINNED_REVISION[:10])))
if not (REVIEW_REPO / '.git').exists():
    subprocess.run(['git','clone','--no-checkout',REPO_URL,str(REVIEW_REPO)],check=True)
    subprocess.run(['git','-C',str(REVIEW_REPO),'checkout','--detach',PINNED_REVISION],check=True)
actual_revision = subprocess.check_output(['git','-C',str(REVIEW_REPO),'rev-parse','HEAD'],text=True).strip()
assert actual_revision == PINNED_REVISION, 'Use the pinned checkout; do not mix revisions.'
PROJECT = REVIEW_REPO / 'Final Project'
EVIDENCE = PROJECT / 'artifacts' / 'corrected'
sys.path.insert(0,str(PROJECT / 'src'))
from eval_utils import evaluate, per_query, tie_keys
from data_utils import short_mask, select_sequence
print('Verified source revision:',actual_revision)


Verified source revision: ced8091063968093c2841460a6f90ddf14e9979b


## 2 Recompute every saved neural metric

The canonical evaluator excludes all-negative query slates, uses gains [0,1,3,7] for NDCG@10, binary relevance for MRR/P@1/HR@10, and candidate-ID-based deterministic tie keys. Current labels are targets for evaluation only. Scores are checked for finiteness and query–candidate keys for duplicates.

In [3]:
pred = pd.read_parquet(EVIDENCE / 'model' / 'test_pred_neural.parquet')
saved = json.loads((EVIDENCE / 'results_neural.json').read_text())
NAMES = {'mlp':'MLP(no towers)','pooled':'Pooled history','short_only':'Short only',
         'order_only':'Order only','dual':'DHAR(ours)','history_cf':'History-conditioned CF'}
assert not pred.duplicated(['session_id','cand_wid']).any()
rows=[]
for variant,name in NAMES.items():
    metrics,n=evaluate(pred,variant+'_score')
    for metric,value in metrics.items():
        error=abs(value-saved[name][metric])
        rows.append(dict(model=name,metric=metric,recomputed=value,saved=saved[name][metric],absolute_difference=error))
        assert error < 1e-9
metric_check=pd.DataFrame(rows)
result_table=metric_check.pivot(index='model',columns='metric',values='recomputed').reindex(NAMES.values())
print('Prediction rows:',len(pred),'query slates:',pred.session_id.nunique(),
      'positive queries:',n,'excluded all-negative queries:',pred.session_id.nunique()-n)
print('Metric values verified:',len(metric_check),'maximum absolute difference:',metric_check.absolute_difference.max())
display(result_table.round(6))


Prediction rows: 266640 query slates: 3000 positive queries: 2965 excluded all-negative queries: 35
Metric values verified: 24 maximum absolute difference: 0.0


metric,HR@10,MRR@200,NDCG@10,P@1
model,,,,
MLP(no towers),0.658010,0.384011,0.372455,0.251939
Pooled history,0.663069,0.384425,0.373802,0.251265
Short only,0.658347,0.382631,0.371341,0.250253
Order only,0.662057,0.382476,0.373051,0.249578
DHAR(ours),0.658347,0.383091,0.371294,0.249916
History-conditioned CF,0.547386,0.271056,0.269364,0.140304


## 3 Independent implementation and row-order invariance
Reimplement the metric arithmetic without calling the repository metric functions. The identity-based tie keys are deliberately shared so tied scores have the same declared ordering. Separately permute every prediction row and require unchanged canonical metrics.

In [4]:
def reference_metrics(frame, score_col):
    frame=frame.copy()
    frame['_key']=tie_keys(frame)
    totals=[]
    for _,group in frame.groupby('session_id',sort=True):
        if not group['label'].gt(0).any(): continue
        ordered=group.sort_values([score_col,'_key'],ascending=[False,True])
        labels=ordered['label'].to_numpy(dtype=int)
        gains=np.array([0.,1.,3.,7.])[labels]
        ideal=np.sort(gains)[::-1]
        k=min(10,len(labels)); discount=np.reciprocal(np.log2(np.arange(2,k+2)))
        ndcg=float(np.dot(gains[:k],discount)/np.dot(ideal[:k],discount))
        relevant=np.flatnonzero(labels[:200]>0)
        totals.append([ndcg,1/(relevant[0]+1) if len(relevant) else 0.,float(labels[0]>0),float(np.any(labels[:10]>0))])
    return dict(zip(['NDCG@10','MRR@200','P@1','HR@10'],np.mean(totals,axis=0)))

shuffled=pred.sample(frac=1,random_state=2026)
checks=[]
for variant,name in NAMES.items():
    canonical,_=evaluate(pred,variant+'_score')
    permuted,_=evaluate(shuffled,variant+'_score')
    independent=reference_metrics(pred,variant+'_score')
    delta=max(abs(canonical[k]-independent[k]) for k in canonical)
    order_delta=max(abs(canonical[k]-permuted[k]) for k in canonical)
    assert delta<1e-12 and order_delta<1e-12
    checks.append(dict(model=name,independent_max_error=delta,row_order_max_error=order_delta))
display(pd.DataFrame(checks))
print('PASS: independent arithmetic and row-order invariance for all six variants.')


PASS: independent arithmetic and row-order invariance for all six variants.


,model,independent_max_error,row_order_max_error
0,MLP(no towers),3.330669e-16,0.0
1,Pooled history,9.992007e-16,0.0
2,Short only,7.216450e-16,0.0
3,Order only,5.551115e-16,0.0
4,DHAR(ours),7.771561e-16,0.0
5,History-conditioned CF,7.771561e-16,0.0


## 4 What does the matched attention comparison establish?

DHAR versus LambdaMART compares model families; it does not isolate attention. DHAR versus the same-feature MLP tests the additional sequence towers more directly. The MLP **still receives engineered historical features** and candidate embeddings. Bootstrap intervals below are conditional on these saved models; they are not variation across training seeds. A confidence interval containing zero is not proof of equivalence.

In [5]:
mlp=per_query(pred,'mlp_score').set_index('session_id')['NDCG@10'].sort_index()
dual=per_query(pred,'dual_score').set_index('session_id')['NDCG@10'].sort_index()
assert mlp.index.equals(dual.index)
delta=dual.to_numpy()-mlp.to_numpy()
rng=np.random.default_rng(42)
boot=np.array([rng.choice(delta,len(delta),replace=True).mean() for _ in range(2000)])
ci=np.quantile(boot,[.025,.975])
reported=json.loads((EVIDENCE/'paired_bootstrap.json').read_text())['dual_minus_mlp']
assert abs(delta.mean()-reported['delta'])<1e-12
assert np.max(np.abs(ci-np.array(reported['ci95'])))<1e-12
print('DHAR minus same-feature MLP NDCG@10:',delta.mean())
print('Paired-query bootstrap 95% interval:',ci)
print('Supported conclusion: this saved run does not establish an incremental benefit from the added attention towers.')
print('Do not assert that feature redundancy, sampling or optimisation caused this result without additional experiments.')


DHAR minus same-feature MLP NDCG@10: -0.0011608994873893593
Paired-query bootstrap 95% interval: [-0.00335616  0.00108642]
Supported conclusion: this saved run does not establish an incremental benefit from the added attention towers.
Do not assert that feature redundancy, sampling or optimisation caused this result without additional experiments.


## 5 Training settings and checkpoint selection

Inspect the recorded training logs rather than inventing neural training-set scores. Hyperparameters are checked against source text; no weight reload or new optimisation is performed. Early stopping selects the best validation epoch and halts after three subsequent non-improvements.

In [6]:
training_rows=[]
for variant,name in NAMES.items():
    log=json.loads((EVIDENCE/f'training_{variant}_seed42.json').read_text())
    best=max(log['epochs'],key=lambda x:x['val_ndcg10'])
    assert abs(best['val_ndcg10']-saved[name+'_val']['NDCG@10'])<1e-9
    training_rows.append(dict(model=name,best_epoch=best['epoch'],epochs_run=len(log['epochs']),
         validation_ndcg10=best['val_ndcg10'],train_slate_width=log['training_slate_width'],
         eval_slate_width=log['evaluation_slate_width'],batch_size=log['batch_size'],device=log['device']))
training_table=pd.DataFrame(training_rows)
display(training_table)
source=(PROJECT/'src'/'step5_neural.py').read_text()
assert 'lr=.002' in source and 'weight_decay=1e-6' in source
assert 'nn.Linear(4*d+nf,128)' in source and 'nn.Linear(128,64)' in source
assert "variant='dual',d=32" in source and 'nn.Dropout(.2)' in source
features=json.loads((EVIDENCE/'feature_summary.json').read_text())['numeric_features']
assert len(features)==26
print('Source-checked: 32-dimensional embeddings; fusion 154; MLP 128-64-1; dropout .2; Adam lr .002, weight decay 1e-6.')
print('Actual feature count:',len(features))


Source-checked: 32-dimensional embeddings; fusion 154; MLP 128-64-1; dropout .2; Adam lr .002, weight decay 1e-6.
Actual feature count: 26


,model,best_epoch,epochs_run,validation_ndcg10,train_slate_width,eval_slate_width,batch_size,device
0,MLP(no towers),2,5,0.367632,64,200,128,cpu
1,Pooled history,2,5,0.367866,64,200,128,cpu
2,Short only,2,5,0.369155,64,200,128,cpu
3,Order only,2,5,0.367846,64,200,128,cpu
4,DHAR(ours),2,5,0.368950,64,200,128,cpu
5,History-conditioned CF,15,18,0.271221,64,200,128,cpu


## 6 Horizon construction and coverage

Short intent = CLICK ≤7 days **or CART ≤90 days**, not all events ≤7 days. Order history = all preceding ORD events. Known-category events are sampled evenly over each complete horizon, up to 20 positions; held-out/unknown categories map to zero and are excluded before sampling. CF uses up to 40 **evenly sampled known-item** positions, not necessarily the latest 40. Chronological sampling is not positional encoding: this attention block does not learn within-sequence position.

The five feature-based variants form the tower comparison. History-conditioned CF uses a different item-history representation without engineered numeric features and is a separate baseline, not the same-input ablation.

In [7]:
boundary=pd.DataFrame({'type':['CLICK','CLICK','CART','CART','ORD','FLW'],
                       'elapsed_days':[7.,7.01,90.,90.01,1.,1.]})
assert short_mask(boundary).tolist()==[True,False,True,False,False,False]
selected=select_sequence(np.arange(101),20)
assert len(selected)==20 and selected[0]==0 and selected[-1]==100
coverage=pd.DataFrame(json.loads((EVIDENCE/'horizon_coverage.json').read_text())).T
coverage['unknown_percent']=100*coverage['unknown_events']/coverage['eligible_events']
coverage['selected_percent_of_eligible']=100*coverage['selected_events']/coverage['eligible_events']
display(coverage.round(3))
print('PASS: action/time boundaries, independent horizon selection policy, and 20-position endpoint coverage.')


PASS: action/time boundaries, independent horizon selection policy, and 20-position endpoint coverage.


,eligible_events,unknown_events,selected_events,users_with_known_sequence,unknown_percent,selected_percent_of_eligible
short,2301544,81178,337562,19569,3.527,14.667
order,785868,69835,237250,18200,8.886,30.190
cf,3091153,369011,631939,19893,11.938,20.443


## 7 Diagram corrections and report wording to review

The existing repository image labels recent history as ≤7 days, omitting the 90-day cart horizon. A revised diagram has been prepared locally for human review; no report or image has been pushed with this notebook.

Source-grounded wording:

- Candidate **category**, not brand or raw query text, supplies each attention query after projection. Brand enters the fused predictor.
- DHAR concatenates category embedding (32), brand embedding (32), short context (32), order context (32), and 26 numeric features: 154 inputs to the 128–64–1 head.
- The pooled variant takes the mean across the concatenated short/order category positions and duplicates that pooled vector into the two context slots. It is not two independently pooled towers.
- The no-tower MLP uses zero context vectors, but retains embeddings and the same engineered history features.
- Full histories contribute to aggregate features. The attention sequences are bounded summaries, not attention over every event.
- Observed action-age windows are consistent with logging differences; JD's actual retention policy is unverified.
- This single-seed result supports a controlled negative finding about added attention, not proof that attention never helps.

Local report replacements, diagram and presentation notes are separate review drafts. Actual human contribution and AI-use declarations must be confirmed by the team; this notebook does not claim that Jiayi authored or personally retrained the existing models.

## 8 Handover status

**Verified in this review:** all 24 saved metrics; six independent metric recomputations; row-order invariance; paired DHAR–MLP interval; training-log best epochs; 26 numeric inputs; attention/head dimensions; action/time boundary semantics; sequence coverage.

**Not performed:** new model training, checkpoint reload without the uncommitted checkpoint files, an independent holdout experiment, a multi-seed run, Google Doc edits, slide edits or final submission.

For fresh training, use the team Step 5 training notebook with Step 2–3 inputs and a separate output directory. Keep source revision, preprocessing, checkpoints and predictions together. Review the report text and diagram before publishing them.